# 模型多模态测试

验证项目涉及的模型对**图片输入（多模态）**的支持，并验证修补后的 `ChatDeepSeekVision` 类。

**背景事实（2026-08-25 实测）：**
- langchain-deepseek 官方文档标注 `ChatDeepSeek` 不支持图片输入（Image input ❌）。该声明来自包内 `data/_profiles.py`（models.dev 生成），其中**未收录** `deepseek-v4-flash-vision-exp` 条目。实测消息层对 `image_url` 块原样透传，DeepSeek 官方 API 可直接识图——官方 ❌ 只是能力档案数据滞后。
- DeepSeek 视觉模型 `deepseek-v4-flash-vision-exp` **仅**在官方端点 `https://api.deepseek.com` 提供；DashScope 未托管（404）。
- 注意：DashScope 上的 deepseek 文本模型收到图片会**静默丢弃**（不报错、不识图），切勿用它做多模态。
- 本项目主对话模型 `qwen3.7-flash` 本身支持多模态；`ChatDeepSeekVision`（`src/agents/harness/orchestration/model_factory.py`）为 DeepSeek 视觉模型补齐了能力档案（`image_inputs=True`）。

**运行环境：agentenv**（`D:\\conda_envs\\agentenv`，已装 `langchain-deepseek` / `langchain-qwq`）。

In [ ]:
import sys
from pathlib import Path
from dotenv import load_dotenv

# 定位 rogers 目录（含 app/config.py），保证可 import app.config / src.agents...
cwd = Path.cwd()
ROGERS = next(
    (p for p in [cwd, *cwd.parents] if (p / "app" / "config.py").exists()),
    cwd,
)
sys.path.insert(0, str(ROGERS))

# 仓库根 .env（与 app/config.py 的 _ENV_FILE 同源）
load_dotenv(ROGERS.parent / ".env")
print("ROGERS:", ROGERS)
print("DEEPSEEK_API_KEY set:", bool(__import__("os").getenv("DEEPSEEK_API_KEY")))
print("DASHSCOPE_API_KEY set:", bool(__import__("os").getenv("DASHSCOPE_API_KEY")))

In [ ]:
import base64
import struct
import zlib


def make_png(width=200, height=200, bg=(200, 30, 30), fg=(30, 30, 200)):
    """生成 200x200 纯色 PNG：红底 + 中央 100x100 蓝色方块（无需 PIL）。"""
    def chunk(tag, data):
        return struct.pack(">I", len(data)) + tag + data + struct.pack(
            ">I", zlib.crc32(tag + data) & 0xFFFFFFFF
        )

    ihdr = struct.pack(">IIBBBBB", width, height, 8, 2, 0, 0, 0)
    x0, x1 = width // 2 - 50, width // 2 + 50
    y0, y1 = height // 2 - 50, height // 2 + 50
    raw = b""
    for y in range(height):
        raw += b"\x00"
        for x in range(width):
            raw += bytes(fg if (x0 <= x < x1 and y0 <= y < y1) else bg)
    return (
        b"\x89PNG\r\n\x1a\n"
        + chunk(b"IHDR", ihdr)
        + chunk(b"IDAT", zlib.compress(raw, 9))
        + chunk(b"IEND", b"")
    )


DATA_URL = "data:image/png;base64," + base64.b64encode(make_png()).decode()
print("data URL length:", len(DATA_URL))

In [ ]:
import os
from langchain_qwq import ChatQwen
from langchain_deepseek import ChatDeepSeek

DASHSCOPE_BASE_URL = "https://dashscope.aliyuncs.com/compatible-mode/v1"

text_messages = [
    (
        "system",
        "You are a helpful assistant that translates English to French. Translate the user sentence.",
    ),
    ("human", "I love programming."),
]

qwen_llm = ChatQwen(
    model="qwen3.7-flash",
    max_tokens=1000,
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=DASHSCOPE_BASE_URL,
)
ds_llm = ChatDeepSeek(
    model="deepseek-v4-flash",
    temperature=0,
    max_tokens=None,
    api_key=os.getenv("DEEPSEEK_API_KEY"),
)

print("Qwen     :", qwen_llm.invoke(text_messages).content)
print("DeepSeek :", ds_llm.invoke(text_messages).content)

## Qwen 多模态（DashScope）

`qwen3.7-flash`（本项目主模型）与 `qwen3-vl-flash`（配置的视觉模型）都支持图片输入。

In [ ]:
multimodal = [
    (
        "human",
        [
            {"type": "text", "text": "描述这张图：背景什么颜色？中间方块什么颜色？"},
            {"type": "image_url", "image_url": {"url": DATA_URL}},
        ],
    )
]

for model in ["qwen3.7-flash", "qwen3-vl-flash"]:
    llm = ChatQwen(
        model=model,
        max_tokens=500,
        api_key=os.getenv("DASHSCOPE_API_KEY"),
        base_url=DASHSCOPE_BASE_URL,
    )
    r = llm.invoke(multimodal)
    print(f"=== {model} ===")
    print("content:", r.content)
    print("usage:", r.usage_metadata)

## DeepSeek 原生 ChatDeepSeek + 视觉模型

官方文档标注不支持图片输入，但实测 `image_url` 块可原样透传、API 正常识图。

In [ ]:
from langchain_deepseek import ChatDeepSeek

ds_vision = ChatDeepSeek(
    model="deepseek-v4-flash-vision-exp",
    temperature=0,
    max_tokens=300,
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    max_retries=2,
)

# 官方能力档案未收录 vision-exp -> profile 为 None（文档 Image input ❌ 的来源）
print("原生 ChatDeepSeek profile:", ds_vision.profile)

r = ds_vision.invoke(multimodal)
print("content:", r.content)
print("usage:", r.usage_metadata)
print("reasoning:", (r.additional_kwargs or {}).get("reasoning_content", "")[:120])

## 修补类 ChatDeepSeekVision（model_factory.py）

继承 `langchain_deepseek.ChatDeepSeek`，补齐 vision-exp 的 `ModelProfile`（`image_inputs=True`），默认走官方端点与 `DEEPSEEK_API_KEY`。

In [ ]:
import importlib.util

# 独立加载 model_factory.py：不经 src.agents 包导入链（该链会拉取后端 DB
# 依赖 asyncpg 等，agentenv 未安装）。model_factory 本身仅依赖 langchain 系。
spec = importlib.util.spec_from_file_location(
    "model_factory",
    ROGERS / "src" / "agents" / "harness" / "orchestration" / "model_factory.py",
)
mf = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mf)

vision = mf.create_deepseek_vision(streaming=True)
print("model:", vision.model_name)
print("profile image_inputs:", (vision.profile or {}).get("image_inputs"))
print("profile:", vision.profile)

# 非流式
r = vision.invoke(multimodal)
print("content:", r.content)
print("usage:", r.usage_metadata)
print("reasoning:", (r.additional_kwargs or {}).get("reasoning_content", "")[:120])

# 流式
parts = []
for chunk in vision.stream(multimodal):
    if chunk.content:
        parts.append(chunk.content)
print("stream:", "".join(parts))


## 负面对照：DashScope 上的 deepseek 文本模型

DashScope 未托管 vision-exp；其上的 `deepseek-v4-flash` 收到图片**静默丢弃**（不报错、`image_tokens=None`、答非所问）。这解释了为什么多模态必须走 DeepSeek 官方端点。

In [ ]:
from openai import OpenAI

client = OpenAI(
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=DASHSCOPE_BASE_URL,
)
r = client.chat.completions.create(
    model="deepseek-v4-flash",
    max_tokens=200,
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "背景什么颜色？"},
                {"type": "image_url", "image_url": {"url": DATA_URL}},
            ],
        }
    ],
)
print("content:", r.choices[0].message.content)
print(
    "image_tokens:",
    r.usage.prompt_tokens_details.image_tokens
    if r.usage.prompt_tokens_details
    else None,
)

## 结论

| 路径 | 图片输入 | 说明 |
| --- | --- | --- |
| `ChatQwen` + `qwen3.7-flash`（DashScope） | ✅ | 本项目主模型原生多模态 |
| `ChatQwen` + `qwen3-vl-flash`（DashScope） | ✅ | 视觉专用，无 reasoning token |
| `ChatDeepSeek` + `deepseek-v4-flash-vision-exp`（官方 API） | ✅ | 官方文档 ❌ 是过时能力档案，实际可用 |
| `ChatDeepSeekVision`（model_factory.py 修补类） | ✅ | 补齐 profile（`image_inputs=True`）+ 官方端点默认值 |
| DashScope + `deepseek-v4-flash` | ⚠️ 静默丢图 | 不报错但不识图，不可用于多模态 |

**生产落点**：`create_deepseek_vision()` 已在 `model_factory.py` 提供，配合 `DEEPSEEK_API_KEY` / `DEEPSEEK_VISION_MODEL` / `DEEPSEEK_TEMPERATURE` 配置（`config.py` + `.env`）。需要环境安装 `langchain-deepseek`（agentenv 已装；生产 venv 未装时 guarded import 自动跳过，不影响其余模型）。